In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
def beam_impl_1(seed=0):

    torch.manual_seed(seed)

    EOS_TOKEN = 0
    PAD_TOKEN = 1

    batch_size = 1
    vocab_size = 8
    k = 4
    alpha = 0.6
    max_len = 20

    #cumulative log probs of each beam
    log_probs = torch.zeros(batch_size, k)  #B, K

    # starting token logprob (probably not correct, every starting beam should have 0)
    a = F.log_softmax(torch.randn(batch_size, vocab_size), dim=-1) #B, VOCAB
    top_a, loc = torch.topk(a, k=k, dim=-1) #B, 4

    # tracks which beams are still active
    active = torch.ones((batch_size, k), dtype=torch.bool)
    active = active & ~(loc == EOS_TOKEN)

    log_probs += top_a

    # Each beam starts with its chosen vocab index
    beams = loc.unsqueeze(-1)

    for i in range(1, max_len):


        new_a = F.log_softmax(torch.randn(batch_size, k, vocab_size), dim=-1)
        
        #always select a finished beam 
        print("new_a shape:", new_a.shape)
        print("active shape:", active.shape)
        new_a[active == False] = float('-inf') 
        new_a[active == False, PAD_TOKEN] = 0.0
        #non finished beams never select PAD_TOKEN
        new_a[active, PAD_TOKEN] = float('-inf')
  

        new_a += log_probs.unsqueeze(-1)
        flat_new_a = new_a.view(batch_size, -1)
        top2_vals, top2_indices = torch.topk(flat_new_a, k=k, dim=-1)

        log_probs = top2_vals
        # Map flat indices back to (beam, token) pairs
        beam_indices = top2_indices // vocab_size
        token_indices = top2_indices % vocab_size

        # Update beams
        batch_idx = torch.arange(batch_size).unsqueeze(1)
        beams = beams[batch_idx, beam_indices, :]
        beams = torch.cat((beams, token_indices.unsqueeze(-1)), dim=-1)
        
        active = active[batch_idx, beam_indices]
        just_finished = token_indices == EOS_TOKEN
        active = active & ~just_finished
        if not active.any():
            break

    # Divide log_probs of each beam by (length - number of pad tokens) ** alpha
    # beams: (batch_size, k, seq_len_so_far)
    # log_probs: (batch_size, k)
    lengths = beams.shape[-1]

    num_pad = (beams == PAD_TOKEN).sum(dim=-1)

    normalized_lengths = (lengths - num_pad).clamp(min=1)

    log_probs = log_probs / (normalized_lengths.float() ** alpha)
    print(log_probs)
    #print(beams)
    return beams
        #check if any beam reached EOS token

In [ ]:
def beam_impl_2(seed=0):

    torch.manual_seed(seed)

    EOS_TOKEN = 0
    PAD_TOKEN = 1

    batch_size = 2
    vocab_size = 8
    k = 5
    alpha = 0.6
    max_len = 20

    #cumulative log probs of each beam
    log_probs = torch.zeros(batch_size, k)  #B, K

    # starting token logprob (probably not correct, every starting beam should have 0)
    # a = F.log_softmax(torch.randn(batch_size, vocab_size), dim=-1) #B, VOCAB
    # top_a, loc = torch.topk(a, k=k, dim=-1) #B, 4

    # tracks which beams are still active
    active = torch.ones((batch_size, k), dtype=torch.bool)
    # active = active & ~(loc == EOS_TOKEN)

    # log_probs += top_a

    # Each beam starts with the token_id 2
    beams = torch.full((batch_size, k, 1), 2.0) #B, K, 1
    
    for i in range(1, max_len):


        new_a = F.log_softmax(torch.randn(batch_size, k, vocab_size), dim=-1)
        
        #always select a finished beam 
        new_a[active == False] = float('-inf') 
        new_a[active == False, PAD_TOKEN] = 0.0
        #non finished beams never select PAD_TOKEN
        new_a[active, PAD_TOKEN] = float('-inf')

        new_a += log_probs.unsqueeze(-1)
        flat_new_a = new_a.view(batch_size, -1)
        top2_vals, top2_indices = torch.topk(flat_new_a, k=k, dim=-1)

        log_probs = top2_vals
        # Map flat indices back to (beam, token) pairs
        beam_indices = top2_indices // vocab_size
        token_indices = top2_indices % vocab_size

        # Update beams
        batch_idx = torch.arange(batch_size).unsqueeze(1)

        print("------------------------------------")
        print(f"beams shape: {beams.shape}")
        print(f"batch_idx shape: {batch_idx.shape}")
        print(f"beam_indices shape: {beam_indices.shape}")
   
        beams = beams[batch_idx, beam_indices, :]
        print(f"new beams shape: {beams.shape}")
   
        beams = torch.cat((beams, token_indices.unsqueeze(-1)), dim=-1)
 
        break
   
        active = active[batch_idx, beam_indices]
        just_finished = token_indices == EOS_TOKEN
        active = active & ~just_finished
        if not active.any():
            break

    # Divide log_probs of each beam by (length - number of pad tokens) ** alpha
    # beams: (batch_size, k, seq_len_so_far)
    # log_probs: (batch_size, k)
    lengths = beams.shape[-1]

    num_pad = (beams == PAD_TOKEN).sum(dim=-1)

    normalized_lengths = (lengths - num_pad).clamp(min=1)

    log_probs = log_probs / (normalized_lengths.float() ** alpha)
    print(log_probs)
    #print(beams)
    return beams
        #check if any beam reached EOS token

In [ ]:
torch.equal(beam_impl_1(),beam_impl_2())